# Construcción de la base final

Preparación reproducible de los datos del MEN sobre deserción escolar en Colombia, 2011 a 2024. Ejecuta las celdas en orden desde la carpeta del proyecto. El CSV original se conserva; las salidas se guardan en `datos/preparados/`.


### Librerías


In [ ]:
import hashlib
import json
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd


## Limpieza de la base


### Definición de campos


In [ ]:
TASAS = ["desercion", "desercion_transicion", "desercion_primaria",
"desercion_secundaria", "desercion_media", "aprobacion", "reprobacion",
"cobertura_neta", "cobertura_bruta"]
BASE = ["ano", "c_digo_departamento", "departamento", "poblacion_5_16"] + TASAS

### Funciones de limpieza


In [ ]:
def numero(serie):
    # El CSV oficial usa punto decimal. Se admite coma decimal sin separador de miles.
    texto = serie.astype("string").str.strip().str.replace(",", ".", regex=False)
    convertido = pd.to_numeric(texto, errors="coerce")
    presentes = texto.notna() & ~texto.isin(["", "nan", "None", "null"])
    if (presentes & convertido.isna()).any():
        raise ValueError(f"Valores no numéricos en {serie.name}; revisar antes de continuar")
    return convertido

def preparar(entrada, salida):
    salida.mkdir(parents=True, exist_ok=True)
    original = pd.read_csv(entrada, dtype="string", keep_default_na=True)
    faltan = set(BASE) - set(original.columns)
    if faltan:
        raise ValueError(f"Faltan columnas requeridas: {sorted(faltan)}")
    duplicados_exactos = int(original.duplicated().sum())
    datos = original.drop_duplicates().loc[:, BASE].copy()
    for c in BASE:
        if c != "departamento":
            datos[c] = numero(datos[c])
    for c in ["ano", "c_digo_departamento", "poblacion_5_16"]:
        if datos[c].isna().any() or ((datos[c] % 1) != 0).any():
            raise ValueError(f"Identificador o población inválido en {c}")
    datos["ano"] = datos["ano"].astype("int64")
    datos["poblacion_5_16"] = datos["poblacion_5_16"].astype("int64")
    datos["c_digo_departamento"] = datos["c_digo_departamento"].astype("int64").astype(str).str.zfill(2)
    if datos["departamento"].isna().any():
        raise ValueError("Departamento ausente")
    antes = datos["departamento"].copy()
    datos["departamento"] = datos["departamento"].str.strip().str.replace(r"\s+", " ", regex=True)
    canon = datos.groupby("c_digo_departamento")["departamento"].agg(lambda s: s.mode().iloc[0]).to_dict()
    canon.update({"11": "Bogotá D.C.", "63": "Quindío"})
    datos["departamento"] = datos["c_digo_departamento"].map(canon)
    corregidos = int((antes != datos["departamento"]).sum())
    if datos.duplicated(["c_digo_departamento", "ano"]).any():
        raise ValueError("Más de un registro por departamento y año; no promediar duplicados conflictivos")
    for c in TASAS:
        v = datos[c].dropna()
        if (v < 0).any() or (c not in ["cobertura_neta", "cobertura_bruta"] and (v > 100).any()):
            raise ValueError(f"Tasa fuera de rango en {c}; revisar el dato original")
    if (datos["poblacion_5_16"] < 0).any():
        raise ValueError("Población negativa")
    # La fuente exporta 3.97 para 3.97 %. Se almacena 0.0397 como proporción para los cálculos y el formato porcentual del tablero.
    datos[TASAS] = datos[TASAS] / 100.0
    datos = datos.sort_values(["c_digo_departamento", "ano"]).reset_index(drop=True)
    datos["rango_desercion"] = np.select(
        [datos.desercion.isna(), datos.desercion < .03, datos.desercion <= .06],
        ["Sin dato", "Baja", "Media"], default="Alta")
    grupo = datos.groupby("c_digo_departamento", sort=False)
    previo = grupo["desercion"].shift(1)
    ano_previo = grupo["ano"].shift(1)
    datos["variacion_interanual_pp"] = ((datos.desercion - previo) * 100).where(datos.ano - ano_previo == 1)
    # Se conservan inconsistencias publicadas: el análisis no inventa correcciones.
    diferencia = (datos.desercion + datos.aprobacion + datos.reprobacion) * 100 - 100
    alertas = datos.loc[diferencia.abs() > .05, ["ano", "c_digo_departamento", "departamento"]].copy()
    alertas["diferencia_suma_pp"] = diferencia[diferencia.abs() > .05].round(6)
    alertas["motivo"] = "Deserción más aprobación más reprobación difiere de 100 % en más de 0.05 pp"
    alertas.to_csv(salida / "alertas_calidad.csv", index=False, encoding="utf-8-sig")
    datos.to_csv(salida / "datos_limpios_powerbi.csv", index=False, encoding="utf-8-sig", float_format="%.8f")
    evolucion = datos.groupby("ano").agg(promedio_desercion=("desercion", "mean"),
        minimo_desercion=("desercion", "min"), maximo_desercion=("desercion", "max"),
        registros=("c_digo_departamento", "size"))
    evolucion.to_csv(salida / "resumen_anual.csv", encoding="utf-8-sig")
    ultimo = int(datos.ano.max())
    actual = datos.loc[datos.ano == ultimo].copy()
    actual.sort_values("desercion", ascending=False).to_csv(salida / "comparacion_ultimo_ano.csv", index=False, encoding="utf-8-sig")
    calidad = {
        "fuente": "https://www.datos.gov.co/resource/ji8i-4anb.csv?$limit=50000",
        "ejecutado_utc": datetime.now(timezone.utc).isoformat(),
        "sha256_original": hashlib.sha256(entrada.read_bytes()).hexdigest(),
        "registros_originales": len(original), "variables_originales": len(original.columns),
        "registros_limpios": len(datos), "variables_limpias": len(datos.columns),
        "periodo": [int(datos.ano.min()), ultimo],
        "territorios_por_codigo": int(datos.c_digo_departamento.nunique()),
        "nombres_distintos_originales": int(original.departamento.nunique()),
        "duplicados_exactos_eliminados": duplicados_exactos,
        "nombres_estandarizados": corregidos,
        "nulos_originales": {k: int(v) for k, v in original.isna().sum().items() if v},
        "nulos_limpios": {k: int(v) for k, v in datos.isna().sum().items() if v},
        "columnas_excluidas_por_alcance": sorted(set(original.columns) - set(BASE)),
        "registros_por_ano": {str(k): int(v) for k, v in datos.groupby("ano").size().items()},
        "coberturas_mayores_a_100_conservadas": {c: int((datos[c] > 1).sum()) for c in ["cobertura_neta", "cobertura_bruta"]},
        "alertas_balance": len(alertas),
        "clasificacion": "Baja <3 %; Media entre 3 y 6 % inclusive; Alta >6 %",
        "unidad_tasas_salida": "proporción; 0.0397 equivale a 3.97 %",
        "unidad_variacion": "puntos porcentuales; no formatear como porcentaje",
        "imputaciones": 0,
        "limitaciones": ["Promedios simples de territorios; no representan una tasa nacional ponderada",
            "Cobertura y deserción no necesariamente comparten universo: la deserción corresponde al sector oficial",
            "Fuente advierte cifras preliminares para 2020 y cambios de población para coberturas 2018 y 2019"]
    }
    (salida / "reporte_calidad.json").write_text(json.dumps(calidad, ensure_ascii=False, indent=2), encoding="utf-8")
    analisis = {
        "ultimo_ano": ultimo,
        "promedio_ultimo": float(actual.desercion.mean()),
        "promedio_primero": float(evolucion.promedio_desercion.iloc[0]),
        "promedio_anterior": float(evolucion.promedio_desercion.iloc[-2]),
        "top": actual.nlargest(5, "desercion")[["departamento", "desercion"]].to_dict("records"),
        "bottom": actual.nsmallest(3, "desercion")[["departamento", "desercion"]].to_dict("records"),
        "niveles": {c: float(actual[c].mean()) for c in TASAS if c.startswith("desercion_")},
        "rangos_ultimo": actual.rango_desercion.value_counts().to_dict(),
        "bajan_ultimo": int((actual.variacion_interanual_pp < 0).sum()),
        "suben_ultimo": int((actual.variacion_interanual_pp > 0).sum()),
        "incrementos": actual.nlargest(3, "variacion_interanual_pp")[["departamento", "variacion_interanual_pp"]].to_dict("records"),
        "reducciones": actual.nsmallest(3, "variacion_interanual_pp")[["departamento", "variacion_interanual_pp"]].to_dict("records"),
        "correlaciones_ultimo": actual[["desercion", "aprobacion", "reprobacion", "cobertura_neta", "cobertura_bruta"]].corr()["desercion"].to_dict(),
        "evolucion": evolucion.reset_index().to_dict("records")
    }
    (salida / "hallazgos.json").write_text(json.dumps(analisis, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({"registros": len(datos), "territorios": calidad["territorios_por_codigo"],
        "periodo": calidad["periodo"], "salida": str(salida)}, ensure_ascii=False))
    return datos, calidad, analisis

## Ejecución de las funciones

La ruta se busca dentro del proyecto para que el notebook funcione en Windows, Linux y Jupyter sin depender de una carpeta personal.


In [ ]:
# Busca la raíz del proyecto desde la carpeta abierta en Jupyter.
carpetas = [Path.cwd(), *Path.cwd().parents]
carpeta_proyecto = next(
    (p for p in carpetas if (p / "datos" / "datos_originales_men.csv").is_file()),
    None,
)
if carpeta_proyecto is None:
    raise FileNotFoundError(
        "Abre Jupyter desde la carpeta del repositorio. "
        "Debe existir datos/datos_originales_men.csv."
    )

entrada = carpeta_proyecto / "datos" / "datos_originales_men.csv"
salida = carpeta_proyecto / "datos" / "preparados"
datos, calidad, analisis = preparar(entrada, salida)
print(f"Proceso terminado. Resultados en: {salida}")
print(f"Registros: {len(datos)}; territorios: {calidad['territorios_por_codigo']}")
datos.head()
